# 03 · Explicit LangGraph workflows

**Prerequisites:** Functions and dictionaries; understand nodes, edges, and state from guide section 7.

**Learning objectives:** Build a stateful graph; use a reducer; enforce evidence checks and a two-draft limit.

**Guide companion:** sections 7 in `LANGCHAIN_LANGGRAPH_LEARNING_GUIDE.md` at the project root.

**How to work:** Run setup, implement each challenge, then run its acceptance cell. Starter functions deliberately raise `NotImplementedError`; this is expected until you complete them. Restart the kernel and run all cells after finishing. You do not need any other notebook or paid API calls. Budget about 45–90 minutes, or longer for the capstone.

Complete solutions are kept in the matching notebook under `solutions/`. There are no hidden solution cells in this notebook.


In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

# Fictional test data, not real people, policies, or research sources.
NOTES = [
    {"id": "s1", "url": "fixture://architecture", "text": "Cedar uses LangGraph to route research tasks."},
    {"id": "s2", "url": "fixture://review", "text": "Cedar pauses its workflow for human review."},
    {"id": "s3", "url": "fixture://ownership", "text": "Mira maintains Cedar."},
    {"id": "s4", "url": "fixture://team", "text": "Mira works on team Atlas."},
    {"id": "s5", "url": "fixture://policy", "text": "Atlas reviews Cedar evidence monthly."},
]
BY_ID = {note["id"]: note for note in NOTES}

from operator import add
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

def fixture_retriever(question):
    # Supplied deterministic dependency: ranking is not this lesson's task.
    ids = {"Who maintains Cedar?": ["s3"], "Cedar": ["s1", "s2", "s3"], "quasar": []}.get(question, [])
    return [dict(BY_ID[source_id]) for source_id in ids]

def supported_writer(evidence, rounds):
    return [{"text": note["text"], "source_id": note["id"]} for note in evidence]

def initial(question):
    return {"question": question, "evidence": [], "claims": [], "issues": [],
            "rounds": 0, "status": "running", "log": []}


## Challenge 1 · Check a draft against evidence

Implement `claim_issues(claims, evidence)`. Return an empty list only when there is at least one claim and every claim exactly matches the text of its cited evidence note. Otherwise return nonempty, readable issue strings. Check against retrieved evidence, not the global collection.
Inputs are well-formed dictionaries; schema validation was covered in notebook 01.


In [ ]:
def claim_issues(claims: list[dict], evidence: list[dict]) -> list[str]:
    raise NotImplementedError("Challenge 1: reject empty or unsupported drafts")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
evidence = [BY_ID["s3"]]
assert claim_issues([{"text": "Mira maintains Cedar.", "source_id": "s3"}], evidence) == []
assert claim_issues([], evidence), "Empty drafts must not pass"
assert claim_issues([{"text": "Cedar is free.", "source_id": "s3"}], evidence)
assert claim_issues([{"text": "Mira maintains Cedar.", "source_id": "missing"}], evidence)
assert claim_issues([{"text": BY_ID["s4"]["text"], "source_id": "s4"}], evidence), "An existing but unretrieved source is not available evidence"
print("PASS: evidence checker")


## Challenge 2 · Build and bound the workflow

Finish the state schema and implement `build_graph(retriever, writer)` returning a compiled LangGraph graph.

- Use nodes named `retrieve`, `draft`, `check`, and `finish`.
- Call `retriever(question)` once. With no results, go directly to `finish`.
- Call `writer(evidence, rounds)` to obtain claims, incrementing `rounds` after each draft.
- Use `claim_issues`; revise invalid drafts while fewer than two drafts have run.
- Finish with `evidence_checked`, `insufficient_evidence`, or `failed_validation`. Clear claims on either failure status.
- Each node appends its own name once to `log`. Use a state reducer; return only new log entries.
- Route with conditional edges. Do not implement the entire workflow as a plain Python loop inside one node.

The test uses a graph recursion limit of 20 as a separate emergency ceiling.


In [ ]:
class WorkflowState(TypedDict):
    question: str
    evidence: list[dict]
    claims: list[dict]
    issues: list[str]
    rounds: int
    status: str
    log: list[str]  # TODO: annotate this field with an accumulation reducer.

def build_graph(retriever, writer):
    raise NotImplementedError("Challenge 2: build the four-node research graph")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
graph = build_graph(fixture_retriever, supported_writer)
assert {"retrieve", "draft", "check", "finish"} <= set(graph.get_graph().nodes)
ok = graph.invoke(initial("Who maintains Cedar?"), {"recursion_limit": 20})
assert ok["status"] == "evidence_checked" and ok["rounds"] == 1
assert ok["claims"] == [{"text": "Mira maintains Cedar.", "source_id": "s3"}]
assert ok["log"] == ["retrieve", "draft", "check", "finish"], "Check your log reducer and updates"
empty = graph.invoke(initial("quasar"), {"recursion_limit": 20})
assert empty["status"] == "insufficient_evidence" and empty["rounds"] == 0
assert empty["claims"] == [] and empty["log"] == ["retrieve", "finish"]

def always_wrong(evidence, rounds):
    return [{"text": "Cedar costs one dollar.", "source_id": "s3"}]

bad = build_graph(fixture_retriever, always_wrong).invoke(initial("Cedar"), {"recursion_limit": 20})
assert bad["status"] == "failed_validation" and bad["rounds"] == 2
assert bad["claims"] == [], "Do not expose failed drafts as final answers"
assert bad["log"] == ["retrieve", "draft", "check", "draft", "check", "finish"]
print("PASS: explicit topology, reducer, success, empty retrieval, and bounded failure")


## Challenge 3 · Repair a failed first draft

Implement `repairing_writer(evidence, rounds)`: on round zero return a claim citing `missing`; on later rounds return exact claims from all supplied evidence. Do not mutate evidence or use global counters.


In [ ]:
def repairing_writer(evidence: list[dict], rounds: int) -> list[dict]:
    raise NotImplementedError("Challenge 3: make the second draft repair the first")


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
repaired = build_graph(fixture_retriever, repairing_writer).invoke(initial("Who maintains Cedar?"))
assert repaired["status"] == "evidence_checked" and repaired["rounds"] == 2
assert repaired["issues"] == [] and repaired["claims"][0]["source_id"] == "s3"
assert repaired["log"].count("retrieve") == 1, "Revision must not rerun retrieval"
print("PASS: a rejected draft can be repaired within the limit")


## Reflection

1. Why is `evidence_checked` a better status than `research_complete` here?
2. What changes if you return the entire previous log from every node?
3. When would a revision need fresh retrieval rather than only a new draft?


**Your answers:**

Write your reasoning here before opening the solutions.


## References

- [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [Graph API examples](https://docs.langchain.com/oss/python/langgraph/use-graph-api)
